In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image

from myDataset import *
from ArchitectureMethods import *
from MetricMethods import *

In [2]:
import torchvision.models as models

# Loads model from .pth file
os.chdir('/user/HS401/ob00564/Documents/COM3001/KDEF/Transfer Learning')

emotionTotal = 7

# Loads MobileNetV2 trained model
mobilenet = models.mobilenet_v2(weights = 'DEFAULT')
mobilenet.classifier = torch.nn.Linear(in_features=1280, out_features= emotionTotal)
mobilenet.load_state_dict(torch.load('KDEF MobileNetV2.pth'))
mobilenet.to('cuda')
mobilenet.eval()

# Loads ResNet18 trained model
resnet18 = models.resnet18(weights = None)
resnet18.fc = nn.Sequential(nn.Linear(resnet18.fc.in_features,emotionTotal))
resnet18.load_state_dict(torch.load('KDEF ResNet18.pth'))
resnet18.to('cuda')
resnet18.eval()

# Loads ResNet34 trained model
resnet34 = models.resnet34(weights = None)
resnet34.fc = nn.Sequential(nn.Linear(resnet34.fc.in_features,emotionTotal))
resnet34.load_state_dict(torch.load('KDEF ResNet34.pth'))
resnet34.to('cuda')
resnet34.eval()
print('Fin')


/tmp/ipykernel_3626275/788366630.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mobilenet.load_state_dict(torch.load('KDEF MobileNetV2.pth'))
/tmp/ipykernel_3626275/78

Fin


In [3]:
from torch.utils.data import ConcatDataset, DataLoader
# Import Datasets: 
os.chdir('/user/HS401/ob00564/Documents/COM3001')

os.chdir('/user/HS401/ob00564/Documents/COM3001/JAFFE')
print(os.getcwd())

jaffe_dataset_tr = myDataset(directory = "DATASET/train", transform = test_transform)
jaffe_dataset_val = myDataset(directory = "DATASET/validation", transform = test_transform)
jaffe_dataset = myDataset(directory = "DATASET/test", transform = test_transform)
compiled_jaffe = ConcatDataset([jaffe_dataset_tr, jaffe_dataset_val, jaffe_dataset] )
jaffe_loader = DataLoader(compiled_jaffe, batch_size = 32, shuffle = False)

os.chdir('/user/HS401/ob00564/Documents/COM3001/KDEF')
print(os.getcwd())

kdef_dataset_tr = myDataset(directory = "DATASET/train", transform = test_transform)
kdef_dataset_val = myDataset(directory = "DATASET/validation", transform = test_transform)
kdef_dataset = myDataset(directory = "DATASET/test", transform = test_transform)
compiled_kdef = ConcatDataset([kdef_dataset_tr, kdef_dataset_val, kdef_dataset])
kdef_loader = DataLoader(compiled_kdef, batch_size = 32, shuffle = False)

os.chdir('/user/HS401/ob00564/Documents/COM3001/CK+')
print(os.getcwd())
ck_dataset_tr = myDataset(directory = "DATASET/train", transform = test_transform)
ck_dataset_val = myDataset(directory = "DATASET/validation", transform = test_transform)
ck_dataset = myDataset(directory = "DATASET/test", transform = test_transform)
compiled_ck = ConcatDataset([ck_dataset_tr, ck_dataset_val, ck_dataset])
ck_loader = DataLoader(compiled_ck, batch_size = 32, shuffle = False)

print(len(compiled_jaffe))
print(len(compiled_kdef))
print(len(compiled_ck))

device = 'cuda'
criterion = nn.CrossEntropyLoss()

/user/HS401/ob00564/Documents/COM3001/JAFFE
/user/HS401/ob00564/Documents/COM3001/KDEF
/user/HS401/ob00564/Documents/COM3001/CK+
213
2938
902


In [4]:
# Testing KDEF Generalizability on JAFFE using MobileNetV2
os.chdir('/user/HS401/ob00564/Documents/COM3001/JAFFE')

print('JAFFE SPLIT on MobileNetV2')
JAFFE_y_true, JAFFE_y_pred, JAFFE_y_score = test(mobilenet, device, criterion, jaffe_loader, 'Test')
class_report(y_pred=JAFFE_y_pred, y_true=JAFFE_y_true)

print('JAFFE SPLIT on ResNet18')
JAFFE_y_true, JAFFE_y_pred, JAFFE_y_score = test(resnet18, device, criterion, jaffe_loader, 'Test')
class_report(y_pred=JAFFE_y_pred, y_true=JAFFE_y_true)

print('JAFFE SPLIT on ResNet34')
JAFFE_y_true, JAFFE_y_pred, JAFFE_y_score = test(resnet34, device, criterion, jaffe_loader, 'Test')
class_report(y_pred=JAFFE_y_pred, y_true=JAFFE_y_true)

JAFFE SPLIT on MobileNetV2
Test Loss: 1.7213, Test Accuracy: 39.44%
              precision    recall  f1-score   support

       Anger     0.1892    0.4667    0.2692        30
     Disgust     0.6250    0.1724    0.2703        29
        Fear     0.5278    0.5938    0.5588        32
   Happiness     0.9167    0.3548    0.5116        31
     Sadness     0.2800    0.2258    0.2500        31
    Surprise     0.6000    0.6000    0.6000        30
     Neurtal     0.3571    0.3333    0.3448        30

    accuracy                         0.3944       213
   macro avg     0.4994    0.3924    0.4007       213
weighted avg     0.5000    0.3944    0.4026       213

JAFFE SPLIT on ResNet18
Test Loss: 1.7918, Test Accuracy: 39.91%
              precision    recall  f1-score   support

       Anger     0.5000    0.1000    0.1667        30
     Disgust     0.8333    0.1724    0.2857        29
        Fear     0.2963    0.7500    0.4248        32
   Happiness     0.8571    0.5806    0.6923        31

In [5]:
os.chdir('/user/HS401/ob00564/Documents/COM3001/CK+')

print('CK+ SPLIT MobileNetV2')
ck_y_true, ck_y_pred, ck_y_score = test(mobilenet, device, criterion, ck_loader, 'Test')
class_report(y_pred=ck_y_pred, y_true=ck_y_true)

print('CK+ SPLIT ResNet18')
ck_y_true, ck_y_pred, ck_y_score = test(resnet18, device, criterion, ck_loader, 'Test')
class_report(y_pred=ck_y_pred, y_true=ck_y_true)

print('CK+ SPLIT ResNet34')
ck_y_true, ck_y_pred, ck_y_score = test(resnet34, device, criterion, ck_loader, 'Test')
class_report(y_pred=ck_y_pred, y_true=ck_y_true)

CK+ SPLIT MobileNetV2
Test Loss: 2.0918, Test Accuracy: 22.28%
              precision    recall  f1-score   support

       Anger     0.0588    0.2889    0.0977        45
     Disgust     0.2581    0.6780    0.3738        59
        Fear     0.0455    0.3200    0.0796        25
   Happiness     0.9231    0.1739    0.2927        69
     Sadness     0.0755    0.5714    0.1333        28
    Surprise     0.7308    0.2289    0.3486        83
     Neurtal     0.9394    0.1568    0.2688       593

    accuracy                         0.2228       902
   macro avg     0.4330    0.3454    0.2278       902
weighted avg     0.7789    0.2228    0.2669       902

CK+ SPLIT ResNet18
Test Loss: 2.2141, Test Accuracy: 42.68%
              precision    recall  f1-score   support

       Anger     0.4444    0.0889    0.1481        45
     Disgust     0.2941    0.5085    0.3727        59
        Fear     0.0452    0.5600    0.0836        25
   Happiness     0.9444    0.2464    0.3908        69
     Sadn

In [6]:
os.chdir('/user/HS401/ob00564/Documents/COM3001/KDEF')

print('KDEF SPLIT MobileNetV2')
KDEF_y_true, KDEF_y_pred, KDEF_y_score = test(mobilenet, device, criterion, kdef_loader, 'Test')

print('KDEF SPLIT ResNet18')
KDEF_y_true, KDEF_y_pred, KDEF_y_score = test(resnet18, device, criterion, kdef_loader, 'Test')

print('KDEF SPLIT ResNet34')
KDEF_y_true, KDEF_y_pred, KDEF_y_score = test(resnet34, device, criterion, kdef_loader, 'Test')

KDEF SPLIT MobileNetV2
Test Loss: 0.4017, Test Accuracy: 86.35%
KDEF SPLIT ResNet18
Test Loss: 0.1208, Test Accuracy: 96.73%
KDEF SPLIT ResNet34
Test Loss: 0.1154, Test Accuracy: 96.77%
